# Préparation des données pour la prédiction salariale

Question : **Peut-on prédire le salaire d'une offre d'emploi en fonction du métier et de l'expérience ?**

Cette étape prépare les variables et le découpage pour l'entraînement et le test. Aucun modèle n'est choisi ou entraîné dans ce notebook.

## 1. Chargement et définition de X et Y

- **Y** : milieu de la fourchette salariale, en USD.
- **X** : intitulé du métier (`job_title`) et niveau d'expérience (`experience_level`).

Les identifiants sont conservés comme index pour retrouver les offres, mais ne sont pas utilisés comme variables. Les bornes du salaire ne sont pas incluses dans X.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / 'data' / 'archive').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'archive'
OUTPUT_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

jobs = pd.read_csv(DATA_DIR / 'fact_job_postings.csv')

required_columns = {
    'job_id', 'job_title', 'experience_level',
    'salary_min_usd', 'salary_max_usd'
}
missing_columns = required_columns - set(jobs.columns)
if missing_columns:
    raise ValueError(f'Colonnes manquantes : {sorted(missing_columns)}')
if jobs['job_id'].isna().any() or jobs['job_id'].duplicated().any():
    raise ValueError('Chaque offre doit avoir un job_id unique et non manquant.')

valid_salary = (
    jobs['salary_min_usd'].notna()
    & jobs['salary_max_usd'].notna()
    & jobs['salary_min_usd'].gt(0)
    & jobs['salary_max_usd'].ge(jobs['salary_min_usd'])
)
eligible_jobs = jobs.loc[valid_salary].copy().set_index('job_id')

X = eligible_jobs[['job_title', 'experience_level']].fillna('Inconnu')
Y = (eligible_jobs['salary_min_usd'] + eligible_jobs['salary_max_usd']) / 2
Y.name = 'salary_midpoint_usd'

print(f'Offres initiales : {len(jobs):,}')
print(f'Offres avec salaire valide : {len(X):,}')
print(f'X : {X.shape[0]:,} lignes, {X.shape[1]} variables')
print(f'Y : {Y.shape[0]:,} valeurs')
display(X.head())
display(Y.head())

## 2. Découpage entraînement / test

On réserve 20 % des offres au test et 80 % à l'entraînement. La graine fixe rend le découpage reproductible. `X` et `Y` sont séparés ensemble afin que chaque cible reste associée à la bonne offre. L'encodage des variables catégorielles sera ajusté plus tard uniquement à partir de l'entraînement, dans le pipeline du modèle choisi.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42
)

print(f'Entraînement : {len(X_train):,} offres ({len(X_train) / len(X):.0%})')
print(f'Test         : {len(X_test):,} offres ({len(X_test) / len(X):.0%})')
print(f'IDs communs entre les deux jeux : {len(set(X_train.index) & set(X_test.index))}')
display(X_train.head())
display(Y_train.head())

## 3. Export des jeux préparés

Les fichiers incluent `job_id` pour la traçabilité. Les variables catégorielles restent textuelles : le modèle choisi ultérieurement devra les encoder en apprenant uniquement sur `X_train`.

In [ ]:
def export_xy(features, target, output_path):
    prepared = features.join(target)
    prepared.index.name = 'job_id'
    prepared.to_csv(output_path, index=True)

export_xy(X_train, Y_train, OUTPUT_DIR / 'salary_train.csv')
export_xy(X_test, Y_test, OUTPUT_DIR / 'salary_test.csv')

print('Fichiers préparés :')
print(OUTPUT_DIR / 'salary_train.csv')
print(OUTPUT_DIR / 'salary_test.csv')